# [1] ???????? ?????? 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# ????????? ??????
data = pd.read_csv('marketing_campaign.csv', sep='\t')

print(f"?????? ????????: {data.shape}")
print(f"???????: {data.columns.tolist()}")


# [2] ??????????? ???? ??????

In [ ]:
# -- ????????? ?????????? ? ??????? ?????????? --
target = 'Response'
print(f"??????? ??????????: {target}")
print(f"?????????? ????????: {data[target].unique()}")
print(f"?????????? ?????????? ????????: {data[target].nunique()}")
print(f"?????????????: {data[target].value_counts().to_dict()}")


?????? ???????? ?????????????. ???????? target Response: 0 ??? 1 - ??? "?????????", ? ?? "???????? ????????". 

# [3] ????????????? ??????

### A: ?????????? ??????? ?? ????????????? (train) ? ???????? (test)

In [ ]:
# --- 1. ???????? ???????? ? ???? ---

# ??????? ??????????
y = data['Response']

# ???????? (??????? ??????? ??????? ? ????? ????????? ????)
X = data.drop(columns=['Response', 'ID', 'Z_CostContact', 'Z_Revenue'])

print("????? X:", X.shape)
print("????? y:", y.shape)


# ---  train/test  ---
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      # 20% ? ????
    random_state=42,    # ????????? ??? ?????????????????
    stratify=y          # ????????? ????????? ??????? 0/1
)

print("Train X:", X_train.shape)
print("Test  X:", X_test.shape)
print("????????????? ??????? ? train:")
print(y_train.value_counts(normalize=True))
print("????????????? ??????? ? test:")
print(y_test.value_counts(normalize=True))


### B: ????????? ???????? ? ??????

In [ ]:
# ---  ?????? ????????? ---

print("???????? ? train:")
print(X_train.isnull().sum())

print("\n???????? ? test:")
print(X_test.isnull().sum())

# ????????? ???????? ?? ?????
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print("\n???????? ????????:", num_cols)
print("?????????????? ????????:", cat_cols)

# --- ?????????? ????????? ---

# 1. ????????: ????????? ???????? ?? train
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='median')
num_imputer.fit(X_train[num_cols])          # ??????? ??????? ?????? ?? train

X_train[num_cols] = num_imputer.transform(X_train[num_cols])
X_test[num_cols]  = num_imputer.transform(X_test[num_cols])

# 2. ?????????????? ????????? ????? (most_frequent)
cat_imputer = SimpleImputer(strategy='most_frequent')
cat_imputer.fit(X_train[cat_cols])

X_train[cat_cols] = cat_imputer.transform(X_train[cat_cols])
X_test[cat_cols]  = cat_imputer.transform(X_test[cat_cols])

# ?????????, ??? ????????? ?????? ???
print("\n????? ?????????, ???????? ? train:")
print(X_train.isnull().sum())
print("\n????? ?????????, ???????? ? test:")
print(X_test.isnull().sum())


### C: ????????????? ????????? ?????????? (StandardScaler)


???? ??????? ?????? ?? train, ????????? ? ? train, ? ? test, ? ??????? ?????? ???????? ???????? (num_cols)

In [ ]:
from sklearn.preprocessing import StandardScaler

# ??????? ???????
scaler = StandardScaler()

# ??????? ????????? (??????? ? std) ?????? ?? train
scaler.fit(X_train[num_cols])

# ????????? ? train ? test
X_train[num_cols] = scaler.transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

X_train[num_cols].head()


### D: ???????????? ?????????????? ???????? ?? ????? ?? ?????????.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = ['Education', 'Marital_Status']  # Dt_Customer ???? ?? ???????

# ?????: ?????????? sparse_output ?????? sparse
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

ohe.fit(X_train[cat_cols])

X_train_cat = ohe.transform(X_train[cat_cols])
X_test_cat  = ohe.transform(X_test[cat_cols])

ohe_feature_names = ohe.get_feature_names_out(cat_cols)

X_train_cat = pd.DataFrame(X_train_cat, columns=ohe_feature_names, index=X_train.index)
X_test_cat  = pd.DataFrame(X_test_cat,  columns=ohe_feature_names, index=X_test.index)

X_train_num = X_train.drop(columns=cat_cols + ['Dt_Customer'])
X_test_num  = X_test.drop(columns=cat_cols + ['Dt_Customer'])

X_train_final = pd.concat([X_train_num, X_train_cat], axis=1)
X_test_final  = pd.concat([X_test_num,  X_test_cat],  axis=1)

X_train_final.head()


# [4] ??????? ?? ????????????? ?????????

### A: ???????? ?????? ((LogisticRegression, LinearRegression))

In [ ]:
from sklearn.linear_model import LogisticRegression

# 1. ??????? ?????? ????????????? ?????????
log_reg = LogisticRegression(
    max_iter=1000,      # ??????????? ????? ????????, ????? ????? ???????
    random_state=42
)

# 2. ??????? ?? ????????????? ??????
log_reg.fit(X_train_final, y_train)

# 3. ?????? ???????????? (???? ??????, ??? ??????)
y_train_pred_lr = log_reg.predict(X_train_final)
y_test_pred_lr  = log_reg.predict(X_test_final)

# ?? ???? 5 ????? ??? ??????? accuracy, f1, roc_auc ? ?.?.
y_train_pred_lr[:10], y_test_pred_lr[:10]


### B: ????????? ?????? (DecisionTreeClassifier, DecisionTreeRegressor)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# DecisionTreeClassifier (?????? ???????)

best_dt = None          # ?????? ?????? ??????
best_dt_score = 0       # ?????? ???????? accuracy ?? ?????

# ?????????? ?????? ???????? ??????? ??????
for depth in [3, 5, 7, 10, 15]:
    # 1. ??????? ?????? ? ???????? ???????????? ????????
    dt = DecisionTreeClassifier(
        max_depth=depth,    # ???????????? ??????? ??????
        random_state=42     # ??? ?????????????????
    )
    
    # 2. ??????? ?????? ?? ????????????? ??????
    dt.fit(X_train_final, y_train)
    
    # 3. ????????? ???????? ?? ???????? ???????
    test_score = dt.score(X_test_final, y_test)
    
    print(f"max_depth={depth}: test_accuracy={test_score:.4f}")
    
    # 4. ???? ??? ?????? ????? ???????????, ?????????? ???
    if test_score > best_dt_score:
        best_dt_score = test_score
        best_dt = dt

# ????? ????? best_dt ? ?????? ??????, best_dt_score ? ??? accuracy ?? ?????
print(f"? DecisionTree ??????? (best depth={best_dt.max_depth}, test_accuracy={best_dt_score:.4f})")


### ?: K-????????? ??????? (KNeighborsClassifier, KNeighborsRegressor) 

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

best_knn = None
best_knn_score = 0

# ?????????? ?????? ???????? k 
for k in [3, 5, 7, 9, 15]:
    knn = KNeighborsClassifier(
        n_neighbors=k,   # ??????? ????????? ??????? ?????????
        weights='distance',  
        metric='minkowski',  
        p=2
    )
    
    # ??????? KNN: ?? ????? ???? ?? ?????? ?????????? ????????? ???????
    knn.fit(X_train_final, y_train)
    
    # ????????? accuracy ?? ?????
    test_score = knn.score(X_test_final, y_test)
    print(f"k={k}: test_accuracy={test_score:.4f}")
    
    # ????????? ?????? ??????
    if test_score > best_knn_score:
        best_knn_score = test_score
        best_knn = knn

print(f"KNN ?????? (best k={best_knn.n_neighbors}, test_accuracy={best_knn_score:.4f})")

### D: ????????? ???

In [ ]:
from sklearn.ensemble import RandomForestClassifier



rf = RandomForestClassifier(
    n_estimators=100,   # ????? ???????? ? ????
    random_state=42,
    n_jobs=-1           # ???????????? ??? ???? CPU
)

# ??????? ??? ?? ????????????? ??????
rf.fit(X_train_final, y_train)

# ????????????? ?????? (0/1)
y_train_pred_rf = rf.predict(X_train_final)
y_test_pred_rf  = rf.predict(X_test_final)

# ????????????? ??????????? ?????? 1 (??? ROC-AUC ? ??.)
y_train_proba_rf = rf.predict_proba(X_train_final)[:, 1]
y_test_proba_rf  = rf.predict_proba(X_test_final)[:, 1]

print("RandomForest ??????? (100 ????????)")


# [5] ?????????? ??????? ?? train ? test

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

results = {}

# 1. LogisticRegression (log_reg ??? ??????? ?????)
results['LogisticRegression'] = {
    'model': log_reg,
    'train_pred':  log_reg.predict(X_train_final),
    'test_pred':   log_reg.predict(X_test_final),
    'train_proba': log_reg.predict_proba(X_train_final)[:, 1],
    'test_proba':  log_reg.predict_proba(X_test_final)[:, 1],
}

# 2. Decision Tree (best_dt ?? ????? ?? max_depth)
results['DecisionTree'] = {
    'model': best_dt,
    'train_pred':  best_dt.predict(X_train_final),
    'test_pred':   best_dt.predict(X_test_final),
    'train_proba': best_dt.predict_proba(X_train_final)[:, 1],
    'test_proba':  best_dt.predict_proba(X_test_final)[:, 1],
}

# 3. KNN (best_knn ?? ????? ?? k)
results['KNN'] = {
    'model': best_knn,
    'train_pred':  best_knn.predict(X_train_final),
    'test_pred':   best_knn.predict(X_test_final),
    'train_proba': best_knn.predict_proba(X_train_final)[:, 1],
    'test_proba':  best_knn.predict_proba(X_test_final)[:, 1],
}

# 4. Random Forest (? ???? ?????? rf)
results['RandomForest'] = {
    'model': rf,
    'train_pred':  rf.predict(X_train_final),
    'test_pred':   rf.predict(X_test_final),
    'train_proba': rf.predict_proba(X_train_final)[:, 1],
    'test_proba':  rf.predict_proba(X_test_final)[:, 1],
}

# === ?????? ?????? ===

metrics_results = {}

for model_name in ['LogisticRegression', 'DecisionTree', 'KNN', 'RandomForest']:
    train_pred  = results[model_name]['train_pred']
    test_pred   = results[model_name]['test_pred']
    train_proba = results[model_name]['train_proba']
    test_proba  = results[model_name]['test_proba']
    
    train_acc = accuracy_score(y_train, train_pred)
    test_acc  = accuracy_score(y_test,  test_pred)
    
    train_auc = roc_auc_score(y_train, train_proba)
    test_auc  = roc_auc_score(y_test,  test_proba)
    
    train_f1 = f1_score(y_train, train_pred)
    test_f1  = f1_score(y_test,  test_pred)
    
    metrics_results[model_name] = {
        'train_acc': train_acc,
        'test_acc':  test_acc,
        'train_auc': train_auc,
        'test_auc':  test_auc,
        'train_f1':  train_f1,
        'test_f1':   test_f1
    }

metrics_df = pd.DataFrame.from_dict(metrics_results, orient='index')
print(metrics_df.round(3))


# [6] ???????? ???????

?????? ??????
? ????? ?????? ??????? ?????? ?? ????? ????? ????? ???????? LogisticRegression: ? ??? ??????? test?accuracy (0.884), ????? ??????? ROC?AUC ?? ????? (0.892) ? ???????? F1 (0.490) ??? ????? ?????????? ???????.?
KNN ? RandomForest ?????????? ???? ???? accuracy, ?? ??????? ???? ROC?AUC ? F1, ???????? ?????? ??????? ?????? ????????? ?? ROC?AUC ? F1 ?? ?????.?

????????????

? ??????: train_acc 0.908 ?????? test_acc 0.864, train_auc 0.839 ?????? test_auc 0.742, train_f1 0.606 ?????? test_f1 0.384 ? ?????? ???? ?????? ????? ?????????? train, ??? ???????? ?? test ? ???????????? ????????????.?

? KNN ? RandomForest: train_acc ? train_f1 ????? 1.0, ROC?AUC ?? train 1.0, ? ?? ????? accuracy ? F1 ??????? ???? ? ???? ????? ???????????? (?????? ????? ???????? ??????? ????????? ???????).?

? ????????????? ????????? ??????? train/test ?? accuracy, ROC?AUC ? F1 ????????? ? ??????????? ????????????, ?????? ???????? ?????????.?

????????????
???????????? ? ????? ? ?? train, ? ?? test ?????????? ??????.?

? ????????????? ????????? train_acc ~0.895 ? train_auc ~0.884 ? ??? ???????? ??????? ????????, ??? ??? ?????? ???????????? ???, ?????? ???????? ?????????? ????? bias ? variance.?

??????/???/KNN ????? ??????????? ????????? ??????? ?? train, ??? ??? ??? ???????????? ???; ???????? ?????? ? ???????????? (variance), ? ?? ? ????????????? ????????? ??????.?

??? ???????? ??????? ???????

??? ??????:

??????? ?????????????: ????????? max_depth, ????????? min_samples_leaf, min_samples_split, ???????????? max_features < 1.0; ??? ?????? ???????????? ? ?????? ??????? test?ROC?AUC ? test?F1.?

??? RandomForest:

????????? ?????????? ??????? ???????? (max_depth, min_samples_leaf), ???????? ? n_estimators (200?500), ???????? ????????? max_features ??? ???????? ???????????? ????????.?

??? KNN:

????????? n_neighbors (????????, 9?31), ????????? ??? ???????? weights='uniform' ? 'distance'; ?????? ??? ??????? k ?????? ?????? ????????????? ? ROC?AUC/F1 ?? ????? ??????.?

??? ????????????? ?????????:

???????? ? ????? ????????????? C (?????? C ? ??????? ?????????????, ?????? C ? ?????? ?????) ? ????? ????????????? (penalty='l1'/'l2'); ????? ????? ??????????? ???????? ????? ??????????? ???????? (????????, ?? ???, ????????? ???????? ? ??????), ??? ????? ???? ??????? ???????, ??? ?????? ?????? ???????????????.